In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd

path = Path("data/Transcripts/AAPL/2016-Apr-26-AAPL.txt")

# Full transcript as one string
text = path.read_text(encoding="utf-8", errors="replace")

# Optional: one row per line for easier preprocessing
transcript_df = pd.DataFrame({"line": text.splitlines()})
print(transcript_df.shape)
transcript_df

(490, 1)


,line
0,
1,
2,Thomson Reuters StreetEvents Event Transcript
3,E D I T E D V E R S I O N
4,
...,...
485,CONFERENCE CALL ITSELF AND THE APPLICABLE COMP...
486,MAKING ANY INVESTMENT OR OTHER DECISIONS.
487,----------------------------------------------...
488,Copyright 2019 Thomson Reuters. All Rights Res...


In [3]:
import json
import re
with open("test_transcript.json", "r") as f:
    data = json.load(f)

text = "Q2 2016 Apple Inc Earnings Call"
match = re.search(r'(Q[1-4])\s(\d{4})\s(.+?)\sEarnings Call', text)

if match:
    quarter = match.group(1)[1]  # Q2
    year = match.group(2)     # 2016
    company = match.group(3)  # Apple Inc
    data["Company"] = company
    data["Year"] = year
    data["Quarter"] = quarter


with open("test_transcript.json", "w") as f:
    json.dump(data, f, indent=4)

In [4]:
name_line_mask = transcript_df['line'].str.match(r'^\s*\*')

# Keep only the participant text after optional whitespace + leading '*'.
participant_names = (
    transcript_df.loc[name_line_mask, 'line']
    .str.replace(r'^\s*\*\s*', '', regex=True)
    .str.strip()
    .reset_index(drop=True)
)
participant_roles = transcript_df.loc[name_line_mask.shift(1, fill_value=False), 'line'].reset_index(drop=True)

pattern = r'^\s*(?P<company>.+?)\s*-\s*(?P<role>.+?)\s*$'
participant_tuples = [
    (name, match.group('company').strip(), match.group('role').strip())
    for name, role_line in zip(participant_names, participant_roles)
    for match in [re.match(pattern, role_line)]
    if match
]

# Normalize company strings so 'Apple Inc' and 'Apple Inc.' match.
def normalize_company_name(value: str) -> str:
    return re.sub(r'\.', '', value or '').strip().casefold()

company_key = normalize_company_name(company)
corporate = [{'Name': entry[0], 'Company': entry[1], 'Role': entry[2]} for entry in participant_tuples if normalize_company_name(entry[1]) == company_key]
analysts = [{'Name': entry[0], 'Company': entry[1], 'Role': entry[2]} for entry in participant_tuples if normalize_company_name(entry[1]) != company_key]

data['Corporate'] = corporate
data['Analysts'] = analysts

with open('test_transcript.json', 'w') as f:
    json.dump(data, f, indent=4)

print('Corporate:', corporate)
print('Analysts:', analysts)


Corporate: [{'Name': 'Luca Maestri', 'Company': 'Apple Inc.', 'Role': 'CFO'}, {'Name': 'Tim Cook', 'Company': 'Apple Inc.', 'Role': 'CEO'}, {'Name': 'Nancy Paxton', 'Company': 'Apple Inc.', 'Role': 'Senior Director of IR'}]
Analysts: [{'Name': 'Katy Huberty', 'Company': 'Morgan Stanley', 'Role': 'Analyst'}, {'Name': 'Gene Munster', 'Company': 'Piper Jaffray & Co.', 'Role': 'Analyst'}, {'Name': 'Rod Hall', 'Company': 'JPMorgan', 'Role': 'Analyst'}, {'Name': 'Shannon Cross', 'Company': 'Cross Research', 'Role': 'Analyst'}, {'Name': 'Toni Sacconaghi', 'Company': 'Bernstein', 'Role': 'Analyst'}, {'Name': 'Simona Jankowski', 'Company': 'Goldman Sachs', 'Role': 'Analyst'}, {'Name': 'Steve Milunovich', 'Company': 'UBS', 'Role': 'Analyst'}]


In [5]:
from speech_parser.speech_isolator import get_speech
rows_between_first_second = get_speech(transcript_df, SPEAKER_1=2, SPEAKER_2=3)
rows_between_first_second

["Thank you. Good afternoon, and thanks to everyone for joining us today. Speaking first is Apple's CEO, Tim Cook, and he'll be followed by CFO, Luca Maestri. After that, we'll open the call to questions from analysts.",
 "Please note that some of the information you'll hear during our discussion will consist of forward-looking statements, including without limitation, those regarding revenue, growth margins, operating expenses, other income and expense, taxes, future business outlook, and plans for capital return and debt issuance. Actual results or trends could differ materially from our forecast.",
 'For more information, please refer to the risk factors discussed in Apples Form 10-K for 2015, the Form 10-Q for the first quarter of FY16, and the Form 8-K filed with the SEC today, along with the Associated Press release. Apple assumes no obligation to update any forward-looking statements or information, which speak as of their respective date.',
 "In addition, today's comments will 

In [6]:
import re
from speech_parser.speaker_isolator import get_speaker

# format 1: 'Full Name, Company - Role [n]'
speaker_pattern = re.compile(
    r"^\s*(?P<name>.+?),\s*(?P<company>.+?)\s*-\s*(?P<role>.+?)\s*\[\d+\]\s*$"
)

# format 2: 'Operator [n]' (or other operator-like speaker labels without company/role)
operator_pattern = re.compile(
    r"^\s*(?P<name>[^,\-\[]+?)\s*\[\d+\]\s*$"
)

speaker_tuples = []
speaker_idx = 1

while True:
    try:
        speaker_line = get_speaker(transcript_df, SPEAKER_INDEX=speaker_idx)
    except IndexError:
        # no more speaker rows
        break
    except ValueError:
        # transcript had no speaker rows
        break

    line = speaker_line.strip()
    match = speaker_pattern.match(line)

    if match:
        speaker_tuples.append(
            (
                match.group("name").strip(),
                match.group("company").strip(),
                match.group("role").strip(),
            )
        )
    else:
        operator_match = operator_pattern.match(line)
        if operator_match:
            speaker_tuples.append(
                (
                    operator_match.group("name").strip(),
                    "",
                    "",
                )
            )

    speaker_idx += 1

print(len(speaker_tuples))
speaker_tuples

52


[('Operator', '', ''),
 ('Nancy Paxton', 'Apple Inc.', 'Senior Director of IR'),
 ('Tim Cook', 'Apple Inc.', 'CEO'),
 ('Luca Maestri', 'Apple Inc.', 'CFO'),
 ('Nancy Paxton', 'Apple Inc.', 'Senior Director of IR'),
 ('Operator', '', ''),
 ('Simona Jankowski', 'Goldman Sachs', 'Analyst'),
 ('Luca Maestri', 'Apple Inc.', 'CFO'),
 ('Tim Cook', 'Apple Inc.', 'CEO'),
 ('Simona Jankowski', 'Goldman Sachs', 'Analyst'),
 ('Nancy Paxton', 'Apple Inc.', 'Senior Director of IR'),
 ('Operator', '', ''),
 ('Gene Munster', 'Piper Jaffray & Co.', 'Analyst'),
 ('Tim Cook', 'Apple Inc.', 'CEO'),
 ('Luca Maestri', 'Apple Inc.', 'CFO'),
 ('Gene Munster', 'Piper Jaffray & Co.', 'Analyst'),
 ('Nancy Paxton', 'Apple Inc.', 'Senior Director of IR'),
 ('Operator', '', ''),
 ('Katy Huberty', 'Morgan Stanley', 'Analyst'),
 ('Luca Maestri', 'Apple Inc.', 'CFO'),
 ('Katy Huberty', 'Morgan Stanley', 'Analyst'),
 ('Tim Cook', 'Apple Inc.', 'CEO'),
 ('Nancy Paxton', 'Apple Inc.', 'Senior Director of IR'),
 ('Operato

In [8]:
from speech_parser.transcript_parser import parse_transcript_to_json
TRANSCRIPT_PATH = 'data/Transcripts/NVDA/2016-Aug-11-NVDA.txt'
parse_transcript_to_json(TRANSCRIPT_PATH)

Corporate Participants


WindowsPath('data/processed/2016-Aug-11-NVDA.json')